# Generate heterostructures

Builds coherent film/substrate interfaces with `pymatgen`'s `CoherentInterfaceBuilder`
and uses Bayesian optimization (`skopt.gp_minimize`) to search over the film's **twist
angle** about the interface normal and the **interlayer gap**, for each available surface
termination pair. Candidates are scored by relaxing them with the MACE potential
(`libraries/model.py`) and computing an adhesion energy, reusing the same
folder/POSCAR/CONTCAR conventions as the rest of this repo (`libraries/utilities.py`).

We only consider coherent interfaces (matching lattices from both sides of the interface,
meaning there is a repeat along the interface that lets us build periodicity along the
surface). Because `substrate_structure`/`film_structure` can have completely different
lattices and symmetries, `CoherentInterfaceBuilder` may only find valid coincidence
(ZSL) matches for specific relative twist angles -- the pre-optimization angle scan below
exists to locate those windows before handing the search to `gp_minimize`.


In [ ]:
import copy
import itertools
import json
import os

import numpy as np
import matplotlib.pyplot as plt

from pymatgen.core.structure import Structure
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.transformations.standard_transformations import RotationTransformation

from ase.io.vasp import write_vasp

from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

import libraries.utilities as sul
import libraries.model      as slm


## Inputs

`CoherentInterfaceBuilder` needs the **bulk** (3-D periodic) unit cells of the substrate
and film -- it cuts its own slabs internally along `substrate_miller`/`film_miller`. Point
these at a relaxed bulk `CONTCAR` (e.g. the one `generate-slabs.ipynb` saves under
`<slab_folder>/bulk/CONTCAR`), **not** at an already-cut slab `CONTCAR` such as
`CONTCAR-0_1_2_i_76` (that naming is the slab-folder convention from
`generate-slabs.ipynb`, which is a strong hint the paths below currently point at slabs
rather than bulk cells -- double-check this).


In [ ]:
# Define name of folder and path to reference POSCAR
general_folder = 'input/CeO2-heterostructure'

# Step 1: Read the bulk structures and the Miller planes to cut them along
substrate_miller = (3, 1, 1)
substrate_POSCAR = "/Users/cibran/work/UPC/SlabOptimization/input/CONTCAR-0_1_2_i_76"
substrate_structure = Structure.from_file(substrate_POSCAR)

film_miller = (4, 4, 3)
film_POSCAR = "/Users/cibran/work/UPC/SlabOptimization/input/CONTCAR-0_1_1_i_14"
film_structure = Structure.from_file(film_POSCAR)


def warn_if_looks_like_slab(structure, label):
    """A structure with a c-axis much longer than a/b is much more likely to be an
    already-cut slab (with vacuum) than a bulk unit cell -- CoherentInterfaceBuilder
    expects the latter."""
    a, b, c = structure.lattice.abc
    if c > 3 * max(a, b):
        print(f"[warning] '{label}' has c={c:.1f} Å vs a/b={a:.1f}/{b:.1f} Å -- this looks "
              f"like an already-cut slab with vacuum, not a bulk unit cell. "
              f"Double check {label}_POSCAR.")


warn_if_looks_like_slab(substrate_structure, 'substrate')
warn_if_looks_like_slab(film_structure, 'film')


## Configuration

In [ ]:
if not os.path.exists(general_folder):
    os.makedirs(general_folder)

# Fixed slab-construction parameters (not searched over -- set these from your own
# convergence tests, the same way min_slab_size/min_vacuum_size are fixed in
# generate-slabs.ipynb)
film_thickness      = 1      # in layer units (in_layers=True below)
substrate_thickness = 1      # in layer units
vacuum_over_film    = 20.0   # Å

# Search space for the Bayesian optimization
angle_bounds = (0.0, 180.0)   # twist angle of the film about the interface normal, degrees
gap_bounds   = (1.5, 4.0)     # interlayer gap, Å

# ZSL matching tolerances. This is the main "different lattice and symmetry" knob: for a
# substrate/film pair with very mismatched lattices or low symmetry you may need a larger
# max_interface_area or looser tolerances before ANY coherent match exists at all.
max_interface_area   = 200.0  # Å², cap on the coincidence supercell area
max_area_ratio_tol    = 0.09
max_length_tol        = 0.03
max_angle_tol         = 0.03
max_matches_per_point = 5     # ZSL matches considered per (termination, angle); the
                               # smallest-area one among these is kept, both because it is
                               # typically the lowest-strain match and because it is the
                               # cheapest to relax with the ML potential

# Bayesian optimization budget (per termination)
n_calls           = 40
n_initial_points  = 10
random_state      = 42
angle_scan_points = 25   # coarse pre-scan used to seed the optimizer, see below

# ML potential settings, matching libraries/model.py. Note libraries/utilities.py's
# relax_structure/read_energy don't currently expose `device`, so this only documents
# what's actually used (model.py defaults to device='cuda'); wire a device kwarg through
# utilities.py if you need to run on CPU.
model_load_path = 'large'

# Worse (J/m²) than any physically expected adhesion energy; returned whenever a candidate
# (angle, gap, termination) does not yield a valid coherent interface, so gp_minimize learns
# to avoid that region without a single failed point derailing the GP surrogate.
PENALTY_ADHESION_ENERGY = 10.0

heterostructure_data = {
    'substrate_miller':    substrate_miller,
    'film_miller':         film_miller,
    'film_thickness':      film_thickness,
    'substrate_thickness': substrate_thickness,
    'vacuum_over_film':    vacuum_over_film,
    'angle_bounds':        angle_bounds,
    'gap_bounds':          gap_bounds,
    'max_interface_area':  max_interface_area,
    'max_area_ratio_tol':  max_area_ratio_tol,
    'max_length_tol':      max_length_tol,
    'max_angle_tol':       max_angle_tol,
    'n_calls':             n_calls,
    'n_initial_points':    n_initial_points,
    'random_state':        random_state,
}

with open(f'{general_folder}/heterostructure_data.json', 'w') as json_file:
    json.dump(heterostructure_data, json_file, indent=2)

# Copy POSCARs there
os.system(f'cp {substrate_POSCAR} {general_folder}/POSCAR-substrate')
os.system(f'cp {film_POSCAR}      {general_folder}/POSCAR-film')


## Interface-building helpers

In [ ]:
zslgen = ZSLGenerator(
    max_area_ratio_tol=max_area_ratio_tol,
    max_area=max_interface_area,
    max_length_tol=max_length_tol,
    max_angle_tol=max_angle_tol,
    bidirectional=True,
)


def build_interface_builder(twist_angle):
    """Rotate a fresh copy of the film about the interface normal (z) by `twist_angle`
    degrees, then build a CoherentInterfaceBuilder against the fixed substrate.

    CoherentInterfaceBuilder has no direct "angle" argument -- it always searches for
    low-strain coincidence (ZSL) matches between the two structures exactly as given.
    Rotating the film changes the *relative* orientation of its in-plane lattice vectors
    against the substrate's, which changes which matches are found. Pre-rotating the film
    like this is how we scan over twist angle.
    """
    rotated_film = copy.deepcopy(film_structure)
    if abs(twist_angle) > 1e-9:
        rotation = RotationTransformation(axis=[0, 0, 1], angle=twist_angle)
        rotated_film = rotation.apply_transformation(rotated_film)

    return CoherentInterfaceBuilder(
        substrate_structure=substrate_structure,
        film_structure=rotated_film,
        film_miller=film_miller,
        substrate_miller=substrate_miller,
        zslgen=zslgen,
    )


def smallest_interface_for_termination(interface_builder, termination, gap):
    """Return the lowest-area Interface for `termination` at the given `gap`, or None if
    the termination isn't available or no coherent match exists."""
    if termination not in interface_builder._terminations:
        return None

    candidates = list(itertools.islice(
        interface_builder.get_interfaces(
            termination=termination,
            gap=gap,
            vacuum_over_film=vacuum_over_film,
            film_thickness=film_thickness,
            substrate_thickness=substrate_thickness,
            in_layers=True,
        ),
        max_matches_per_point,
    ))
    if not candidates:
        return None

    areas = [np.linalg.norm(np.cross(itf.lattice.matrix[0], itf.lattice.matrix[1])) for itf in candidates]
    return candidates[int(np.argmin(areas))]


def to_plain_structure(structure):
    """Interface objects carry extra site_properties (Wyckoff labels, film/substrate
    tags) that some ASE/pymatgen version combinations choke on when converting to
    ase.Atoms. Stripping to a plain Structure with just species/coords avoids that."""
    return Structure(structure.lattice, structure.species, structure.frac_coords, coords_are_cartesian=False)


def write_structure(structure, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    write_vasp(path, AseAtomsAdaptor.get_atoms(to_plain_structure(structure)), direct=True, sort=True)


## Adhesion energy

Analogous to `sul.get_surface_energy_of_formation`, but normalized by a single interface
(not the two free surfaces of an isolated slab):

\begin{equation}
    E_{\text{adhesion}} = \frac{E_{\text{interface}} - E_{\text{film}} - E_{\text{substrate}}}{S}
\end{equation}

where all energies are total energies (eV) of structures sharing the same (strained)
interface cell -- `interface.film`/`interface.substrate` give exactly that -- and the
result is converted to J/m² with the same eV/Å² factor used elsewhere in this repo. A
negative value means the interface is more stable than the two isolated, independently
relaxed slabs.


In [ ]:
def get_adhesion_energy_of_formation(interface_energy, film_energy, substrate_energy, interface_area):
    return (interface_energy - film_energy - substrate_energy) * 16.0218 / interface_area


In [ ]:
def evaluate_candidate(twist_angle, gap, termination, eval_folder):
    """Build the interface for (twist_angle, gap, termination), relax interface/film/
    substrate with the MACE potential via the existing libraries.utilities/model
    pipeline, and return the adhesion energy in J/m² (or None if no coherent interface
    exists for this point)."""
    interface_builder = build_interface_builder(twist_angle)
    interface = smallest_interface_for_termination(interface_builder, termination, gap)
    if interface is None:
        return None

    interface_area = np.linalg.norm(np.cross(interface.lattice.matrix[0], interface.lattice.matrix[1]))

    interface_folder = f'{eval_folder}/interface'
    film_folder       = f'{eval_folder}/film'
    substrate_folder  = f'{eval_folder}/substrate'

    write_structure(interface,           f'{interface_folder}/POSCAR')
    write_structure(interface.film,      f'{film_folder}/POSCAR')
    write_structure(interface.substrate, f'{substrate_folder}/POSCAR')

    interface_energy = sul.read_energy(interface_folder, model_load_path=model_load_path)
    film_energy      = sul.read_energy(film_folder,      model_load_path=model_load_path)
    substrate_energy = sul.read_energy(substrate_folder, model_load_path=model_load_path)

    if any(e is None or np.isnan(e) for e in (interface_energy, film_energy, substrate_energy)):
        return None

    adhesion_energy = get_adhesion_energy_of_formation(interface_energy, film_energy, substrate_energy, interface_area)

    with open(f'{eval_folder}/candidate_data.json', 'w') as json_file:
        json.dump({
            'termination':     [str(t) for t in termination],
            'twist_angle':     twist_angle,
            'gap':             gap,
            'interface_area':  interface_area,
            'interface_energy': interface_energy,
            'film_energy':      film_energy,
            'substrate_energy': substrate_energy,
            'adhesion_energy':  adhesion_energy,
        }, json_file, indent=2)

    return adhesion_energy


## Seeding the search

A plain `gp_minimize` over a continuous twist angle will mostly land on angles with **no**
coherent match at all (a flat `PENALTY_ADHESION_ENERGY`), since coincidence matches only
exist at specific relative orientations set by each pair of lattices. A Gaussian-process
surrogate has little to work with when most of the space is flat, so we first run a cheap,
ML-potential-free geometry scan to find angles that actually admit a match, and seed
`gp_minimize`'s initial points with those.


In [ ]:
def find_valid_twist_angles(termination, gap_for_scan, n_scan=angle_scan_points):
    valid_angles = []
    for angle in np.linspace(angle_bounds[0], angle_bounds[1], n_scan):
        interface_builder = build_interface_builder(float(angle))
        if smallest_interface_for_termination(interface_builder, termination, gap_for_scan) is not None:
            valid_angles.append(float(angle))
    return valid_angles


## Run the optimization, per termination

In [ ]:
# Reference (unrotated) builder, just to enumerate the terminations available for this
# substrate/film + Miller-plane combination. Terminations come from how each bulk
# structure is cut along its own Miller plane, so -- unlike the ZSL matches -- they don't
# depend on twist_angle.
reference_builder = build_interface_builder(twist_angle=0.0)
terminations = list(reference_builder._terminations.keys())
print(f'Found {len(terminations)} termination combination(s): {terminations}')

results = {}
for t_idx, termination in enumerate(terminations):
    termination_str = f'termination-{t_idx}'
    termination_folder = f'{general_folder}/{termination_str}'
    print(f'\n=== {termination_str}: {termination} ===')

    seed_angles = find_valid_twist_angles(termination, gap_for_scan=float(np.mean(gap_bounds)))
    print(f'{len(seed_angles)}/{angle_scan_points} scanned angles admit a coherent match')
    if not seed_angles:
        print('No coherent match found anywhere in angle_bounds for this termination -- '
              'skipping. Consider widening angle_bounds, max_interface_area, or the ZSL '
              'tolerances above.')
        continue

    # gp_minimize requires n_calls >= n_initial_points + len(x0); subsample the seed angles
    # (evenly across the valid range) if the scan found more of them than the budget allows
    max_seed_points = max(0, n_calls - n_initial_points)
    if len(seed_angles) > max_seed_points:
        idx = sorted(set(np.linspace(0, len(seed_angles) - 1, max_seed_points).round().astype(int)))
        seed_angles = [seed_angles[i] for i in idx]

    x0 = [[angle, float(np.mean(gap_bounds))] for angle in seed_angles]
    call_counter = itertools.count()

    dimensions = [
        Real(*angle_bounds, name='twist_angle'),
        Real(*gap_bounds, name='gap'),
    ]

    @use_named_args(dimensions)
    def energy_function(twist_angle, gap, termination=termination, termination_folder=termination_folder):
        call_idx = next(call_counter)
        eval_folder = f'{termination_folder}/eval_{call_idx}'
        adhesion_energy = evaluate_candidate(twist_angle, gap, termination, eval_folder)
        if adhesion_energy is None:
            return PENALTY_ADHESION_ENERGY
        print(f'  call {call_idx}: angle={twist_angle:.2f}°, gap={gap:.2f} Å -> '
              f'E_adhesion={adhesion_energy:.4g} J/m²')
        return adhesion_energy

    res = gp_minimize(
        energy_function,
        dimensions=dimensions,
        n_calls=n_calls,
        n_initial_points=n_initial_points,
        x0=x0,
        random_state=random_state,
    )

    results[termination_str] = {
        'termination':          [str(t) for t in termination],
        'best_twist_angle':     float(res.x[0]),
        'best_gap':             float(res.x[1]),
        'best_adhesion_energy': float(res.fun),
    }
    print(f'Best for {termination_str}: angle={res.x[0]:.2f}°, gap={res.x[1]:.2f} Å, '
          f'E_adhesion={res.fun:.4g} J/m²')

with open(f'{general_folder}/heterostructure_results.json', 'w') as json_file:
    json.dump(results, json_file, indent=2)

results


## Best interface overall

In [ ]:
if not results:
    raise RuntimeError('No termination produced a valid coherent interface anywhere in the search space.')

best_termination_str = min(results, key=lambda k: results[k]['best_adhesion_energy'])
best = results[best_termination_str]
print(f'Overall best: {best_termination_str} {best["termination"]}  '
      f'angle={best["best_twist_angle"]:.2f}°, gap={best["best_gap"]:.2f} Å, '
      f'E_adhesion={best["best_adhesion_energy"]:.4g} J/m²')

best_folder = f'{general_folder}/best'
os.makedirs(best_folder, exist_ok=True)

term_idx = int(best_termination_str.split('-')[1])
best_termination = terminations[term_idx]

interface_builder = build_interface_builder(best['best_twist_angle'])
best_interface = smallest_interface_for_termination(interface_builder, best_termination, best['best_gap'])
write_structure(best_interface, f'{best_folder}/POSCAR')

with open(f'{best_folder}/best_data.json', 'w') as json_file:
    json.dump(best, json_file, indent=2)


## Ranking plot

In [ ]:
labels   = list(results.keys())
energies = [results[k]['best_adhesion_energy'] for k in labels]

plt.figure(figsize=(max(4, len(labels) * 1.2), 4))
plt.plot(energies, 'o-')
plt.xticks(range(len(labels)), labels, rotation=45, ha='right')
plt.ylabel(r'$E_{\text{adhesion}}$ (J/m$^2$)')
plt.axhline(0, color='grey', linewidth=0.8)
plt.tight_layout()
plt.savefig(f'{general_folder}/adhesion_ranking.png', dpi=200)
plt.show()
